In [131]:
import numpy as np

In [150]:
def mini_batching(batch_size, X_train):
    batch = []
    batch_count = 0
    batch_last = []
    for i in range(len(X_train) // batch_size):
        batch.append(X_train[batch_size*i:batch_size*(i+1)])
        batch_count += 1
        print(len(X_train[batch_size*i:batch_size*(i+1)]))
    batch_last = X_train[batch_count*batch_size:]
    if len(batch_last) != 0:
        batch.append(batch_last)
    return batch

In [154]:
def initialize_parameters(layer_dims):
    parameters = {}
    L = len(layer_dims)
    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l - 1]) * np.sqrt(2) / layer_dims[l - 1] 
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
        parameters['g' + str(l)] = np.ones((layer_dims[l], 1))
        parameters['beta' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

In [134]:
def batch_norm_forward(Z, gamma, beta, epsilon = 1e-7):
    mu = np.mean(Z, axis = 1, keepdims = True)
    xmu = Z - mu
    var = np.mean(xmu**2, axis = 1, keepdims = True)
    sqrt_var = np.sqrt(var + epsilon)
    ivar = 1.0 / sqrt_var

    Z_norm = xmu * ivar

    Z_tilde = gamma * Z_norm + beta

    cache = (Z_norm, xmu, ivar, sqrt_var, var, mu, gamma)
    return Z_tilde, gamma

In [135]:
def forward_pass(X, parameters, batch_norm_forward):
    cache = {}
    cache['A0'] = X
    L = len(parameters) // 4
    for l in range(1, L):
        Z = np.dot(parameters['W' + str(l)], cache['A' + str(l - 1)]) + parameters['b' + str(l)]
        Z_tilde, b_cache = batch_norm_forward(Z, parameters['gamma' + str(l)], parameters['beta' + str(l)])
        cache['Z' + str(l)] = Z
        cache['Z_tilde' + str(l)] = Z_tilde
        cache['b_cache' + str(l)] = b_cache
        cache['A' + str(l)] = np.maximum(0, Z_tilde)
    ZL = np.dot(parameters['W' + str(l)], cache['A' + str(L)]) + parameters['b' + str(L)]
    cache['Z' + str(L)] = ZL
    cache['A' + str(L)] = 1 / (1 + np.exp(-Z))
    AL = cache['A' + str(L)]
    return AL, cache

In [137]:
def batch_backprop(dZ_tilde, cache, parameters):
    Z_norm, xmu, ivar, sqrtvar, var, mu, gamma = cache
    m = dZ_tilde.shape[1]

    dbeta = np.sum(dZ_tilde, axis = 1, keepdims = True)
    dgamma = np.sum(dZ_tilde * Z_norm, axis = 1, keepdims = True)
    dZ_norm = dZ_tilde * gamma

    divar = np.sum(dZ_norm * xmu, axis = 1, keepdims= True)
    dsqrtvar = divar * (- 1.0 / np.square(sqrtvar))
    dvar = np.sum(dsqrtvar) * (1.0 / (2 * sqrtvar))
    dxmu = dZ_norm * ivar + (2 * dvar * xmu) / m
    dmu = np.sum(dxmu * (- 1.0), axis = 1, keepdims = True)

    dZ = dxmu + dmu * 1/m

    return dZ, dgamma, dbeta

In [138]:
def backprop(X, Y, parameters, AL, cache):
    m = Y.shape[1]
    L = len(parameters) // 4
    grads = {}
    dZL =  AL - Y
    grads['dW' + str(L)] = (1.0 / m) * np.dot(dZL, cache['A' + str(L - 1)].T)
    grads['db' + str(L)] = (1.0 / m) * np.sum(dZL, axis = 1, keepdims = True)
    grads['dA' + str(L - 1)] = np.dot(parameters['W' + str(L)].T, dZL)
    for l in reversed(range(1, L)):
        dZ_tilde = np.copy(grads['dA' + str(l)])
        dZ_tilde[cache['Z_tilde' + str(l)] <= 0] = 0
        grads['dZ_tilde' + str(l)] = dZ_tilde
        dZ, dgamma, dbeta = batch_backprop(grads['dZ_tilde' + str(l)], cache['b_cache' + str(l)], parameters)
        grads['dW' + str(l)] = (1.0 / m) * np.dot(grads['dZ' + str(l)], cache['A' + str(l - 1)].T)
        grads['dA' + str(l - 1)] = np.dot(parameters['W' + str(l)].T, grads['dZ' + str(l)])

    return grads

In [139]:
def update_params(parameters, grads, learning_rate= 0.01):
    L = len([k for k in parameters.keys() if k.startswith('W')])
    for l in range(1, L + 1):
        parameters['W' + str(l)] -= learning_rate * grads['dW' + str(l)]
        parameters['b' + str(l)] -= learning_rate * grads['db' + str(l)]
        if l < L:
            parameters['gamma' + str(l)] -= learning_rate * grads['dgamma' + str(l)]
            parameters['beta' + str(l)] -= learning_rate * grads['dbeta' + str(l)]
            
    return parameters

In [151]:
def compute_cost(AL, Y):
    m = Y.shape[1]
    AL = np.clip(AL, 1e-15, 1.0 - 1e-15)
    cost = (- 1.0 / m) * np.sum(Y*np.log(AL) + (1 - Y) * np.log(1 - AL))
    return np.squeeze(cost)

In [140]:
from sklearn.datasets import make_moons

In [141]:
X_raw, Y_raw = make_moons(500, noise = 0.2, random_state = 42)

In [147]:
from sklearn.model_selection import train_test_split
X_train_raw, X_test_raw, Y_train_raw, Y_test_raw = train_test_split(X_raw, Y_raw, test_size = 0.2, random_state = 42)
X_train = X_train_raw.T
X_test = X_test_raw.T
Y_train = Y_train_raw.reshape(1, -1)
Y_test = Y_test_raw.reshape(1, -1)

In [152]:
def model(X, Y, layer_dims, num_epochs = 1000, batch = 64, learning_rate = 0.01):
    parameters = initialize_parameters(layer_dims)
    for i in range(num_epochs):
        minibatches = mini_batching(X, Y, batch)
        for minibatch in minibatches:
            X_mini, Y_mini = minibatch
            AL, cache = forward_pass(X_mini, parameters, batch_norm_forward)
            batch_cost = compute_cost(AL, Y_mini)
            epoch_cost = batch_cost / batch
            grads = backprop(X_mini, Y_mini, parameters, AL, cache)
            parameters = update_params(parameters, grads, learning_rate)
        if i % 100 == 0:
            print(f'Cost after iteration {i} is {epoch_cost}')
    return parameters